# DEXTRA · 11 — The fair comparison, and the CPCB sub-index head

[![Open In Colab](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/11_fair_comparison.ipynb)](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/11_fair_comparison.ipynb)

Notebook 08 compared the final model against baselines that were **not given the same
inputs**. The deep baselines read the 26 daily variates; the final model read those plus
the evening features, the nightly anchors and 72 hours of sub-daily shape. A reviewer who
opens the code will see that, so this notebook removes the confound.

Two things happen here.

**1 · The fair comparison.** Every *multivariate* architecture is re-run on one identical
feature set — the daily variates, the evening features, the nightly anchors **and** the
same `h_*` hourly-shape columns that the gradient-boosting rival already receives. If the
all-token head still wins, there is an architecture result. If it does not, we report that.

A note on DLinear and PatchTST: both are **univariate by construction** in this
repository, and in their original papers — their `forward` reads only
`window[:, :, target_index]`, the AQI history alone. Handing them more columns would
change nothing, so they are not re-run. The paper must state that they are univariate
baselines rather than implying they were starved of features.

**2 · The CPCB sub-index head.** A new rung: instead of predicting the AQI, the model
predicts the pollutant **concentrations** and reads the AQI off the CPCB formula itself,
written as a differentiable layer. The loss then back-propagates through *which pollutant
dominates*. Nothing in `src/models/` does this today.

**This must run before any 2026 value is read.** `PREREGISTRATION.md` forbids model
changes after that point, and nothing of the 2026 test has been run, so we are still free.
The notebook refuses to start if 2026 data has reached the dataset.

In [ ]:
# ── config ──────────────────────────────────────────────────────────────────
HF_DATASET_REPO = ""          # leave empty to read the HF_DATASET_REPO Colab secret
GITHUB_REPO_URL = "https://github.com/Aduitya9/DEXTRA.git"

SEEDS = (42, 43, 44)          # the same three seeds notebook 08 used
EPOCHS = 60
PATIENCE = 8
BATCH_SIZE = 256
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
DROPOUT = 0.3
D_MODEL, N_HEADS, N_LAYERS = 128, 8, 2

TIME_BUDGET_HOURS = 10.5      # stop cleanly before Colab does
LOCO_CITIES = None            # None = all fifteen; or e.g. ["Kolkata"] for a quick look
SAVE_TO_HF = True             # False = run without writing anything back

### If Colab disconnects

Nothing is lost. Every city is recorded on HuggingFace the moment it finishes, in its own
commit. Re-open this notebook, **Run all**, and it skips what is already done and carries
on from the first unfinished city. A city interrupted part-way is simply redone.

Expect roughly **two to four hours** on an L4 for all fifteen cities — notebook 08
recorded about seven minutes per city for twenty-one fits, and this is fifteen fits per
city on a wider feature set. If the session ends first, run it again; it picks up where
it stopped.

In [ ]:
# ── secrets: Colab if available, environment otherwise ──────────────────────
import os

HF_TOKEN = None
try:
    from google.colab import userdata
    IN_COLAB = True
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        print("! No HF_TOKEN secret. Reading is fine (the dataset is public) but nothing "
              "can be saved. Key icon in the sidebar -> add HF_TOKEN -> enable for this notebook.")
    if not HF_DATASET_REPO:
        try:
            HF_DATASET_REPO = userdata.get("HF_DATASET_REPO")
        except Exception:
            pass
except ImportError:
    IN_COLAB = False
    HF_TOKEN = os.environ.get("HF_TOKEN")
    HF_DATASET_REPO = HF_DATASET_REPO or os.environ.get("HF_DATASET_REPO", "")

if not HF_DATASET_REPO:
    HF_DATASET_REPO = "Aduitya9/Aduitya_dextra-aqi-india"
if SAVE_TO_HF and not HF_TOKEN:
    SAVE_TO_HF = False
    print("! No token, so SAVE_TO_HF is off: this run will compute but not record.")
print(f"dataset repo: {HF_DATASET_REPO} | colab={IN_COLAB} | saving={SAVE_TO_HF}")

In [ ]:
if IN_COLAB:
    get_ipython().run_line_magic(  # noqa: F821
        "pip", 'install -q "huggingface_hub>=0.23.0" pyarrow')

In [ ]:
# ── setup: the repo, the data, and the one feature set everything will share ─
import hashlib
import io
import subprocess
import sys
import tempfile
import time
from pathlib import Path

REPO_DIR = "/content/DEXTRA" if IN_COLAB else os.path.abspath("DEXTRA")
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", GITHUB_REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from huggingface_hub.utils import disable_progress_bars
from src import config
from src.data import assemble, evening, hourly
from src.evaluation import gap, noise
from src.models import deep_baselines, itransformer, multires, sequences, training
from src.utils import aqi_calculator
from src.utils.hf_store import HFStore

disable_progress_bars()
device = itransformer.pick_device()
store = HFStore(HF_DATASET_REPO, token=HF_TOKEN)


def by_day(frame):
    frame["date"] = pd.to_datetime(frame["date"]).astype("datetime64[ns]")
    return frame


LOOKBACK, HORIZON = config.LOOKBACK_WINDOW, config.FORECAST_HORIZON
ANCHOR_COLUMNS = ["anchor1", "anchor2", "anchor3"]
KEYS = ["city", "date", "horizon"]
LINE = config.RARE_AQI_THRESHOLD

hourly_table = store.read_df("processed/city_hourly.parquet")
hourly_table["timestamp"] = pd.to_datetime(hourly_table["timestamp"])
daily = by_day(store.read_df("processed/city_daily_features.parquet"))
daily = daily.merge(by_day(store.read_df("processed/evening_features.parquet")),
                    on=["city", "date"], how="left")
daily = hourly.add_night_anchors(daily, hourly_table)

# The SAME hourly-shape columns the gradient-boosting rival already gets. This is what
# makes the comparison fair: every multivariate model now sees the hourly shape too.
shape = by_day(store.read_df("processed/hourly_dynamics.parquet"))
SHAPE_COLUMNS = [c for c in shape.columns if c.startswith("h_")]
daily = daily.merge(shape[["city", "date"] + SHAPE_COLUMNS], on=["city", "date"], how="left")

FEATURES = {"daily": list(config.ALL_FEATURES)}
FEATURES["evening"] = FEATURES["daily"] + evening.FEATURES
FEATURES["final"] = FEATURES["evening"] + ANCHOR_COLUMNS
FEATURES["fair"] = FEATURES["final"] + SHAPE_COLUMNS       # one set, every model

variates = {kind: sequences.feature_names(daily, columns) for kind, columns in FEATURES.items()}
target = {kind: names.index(sequences.TARGET) for kind, names in variates.items()}

# 2026 is sealed until the model is frozen: see PREREGISTRATION.md.
SEAL = pd.Timestamp("2026-01-01")
if daily.date.max() >= SEAL or hourly_table.timestamp.max() >= SEAL:
    raise RuntimeError("2026 data reached this notebook. It is sealed for the final test.")

try:
    session_age = float(open("/proc/uptime").read().split()[0])
except (OSError, ValueError, IndexError):
    session_age = 0.0
if not 0.0 <= session_age < 12 * 3600:
    session_age = 0.0
deadline = time.monotonic() + max(0.0, TIME_BUDGET_HOURS * 3600 - session_age)

registry = training.RunRegistry(store, "runs/11")
registry08 = training.RunRegistry(store, "runs/08")
cities = sorted(daily.city.unique()) if LOCO_CITIES is None else list(LOCO_CITIES)
fair_run = lambda city: "loco-" + city.replace(" ", "_") + "-fair"
subindex_run = lambda city: "loco-" + city.replace(" ", "_") + "-subindex"
planned = [run(c) for run in (fair_run, subindex_run) for c in cities]
finished = [name for name in planned if registry.done(name)]
seeded = lambda name: [f"{name}_seed{s}" for s in SEEDS]

print(f"device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
if device.type != "cuda":
    print("! No GPU: this will be very slow. Runtime -> Change runtime type -> L4 GPU, then Run all.")
print(f"{len(daily):,} city-days | {len(SHAPE_COLUMNS)} hourly-shape columns | "
      f"shared feature set 'fair' is {len(variates['fair'])} variates")
print(f"{len(finished)} of {len(planned)} pieces already finished"
      + (" -- resuming where the last session stopped" if finished else ""))
print(f"time budget left this session: {(deadline - time.monotonic()) / 3600:.1f} h")

## A · The CPCB sub-index as a differentiable layer

The CPCB sub-index for a pollutant is a monotone piecewise-linear function of
concentration, fixed by a published breakpoint table. Written as a sum of clamped ramps
it is differentiable almost everywhere:

$$I(c) = \sum_k \Delta I_k \cdot \mathrm{clamp}\!\left(\frac{c - C_k}{C_{k+1}-C_k},\,0,\,1\right)$$

The index is then the maximum over pollutants. A hard maximum passes gradient to one
pollutant only, so we use a straight-through estimator: the **value** is the exact
maximum, and the **gradient** is that of a softened maximum, which reaches every
pollutant near the top. The breakpoints are read from `src/utils/aqi_calculator.py`, so
this layer cannot drift from the calculator the dataset itself used.

In [ ]:
class CPCBSubIndex(nn.Module):
    """The CPCB National AQI formula as a differentiable output layer."""

    def __init__(self, pollutants, tau=1.0, cap=500.0):
        super().__init__()
        self.pollutants, self.tau, self.cap = list(pollutants), float(tau), float(cap)
        knots_c, knots_i = [], []
        for name in self.pollutants:
            table = aqi_calculator.BREAKPOINTS[name]
            # knots at 0 and at each bracket's upper bound; values at 0 and each I_high
            knots_c.append([0.0] + [float(b[1]) for b in table])
            knots_i.append([0.0] + [float(b[3]) for b in table])
        c = torch.tensor(knots_c, dtype=torch.float32)          # (P, K+1)
        i = torch.tensor(knots_i, dtype=torch.float32)
        self.register_buffer("lo", c[:, :-1])                    # (P, K)
        self.register_buffer("width", (c[:, 1:] - c[:, :-1]).clamp(min=1e-6))
        self.register_buffer("step", i[:, 1:] - i[:, :-1])

    def sub_index(self, conc):
        """(..., P) concentrations -> (..., P) sub-indices."""
        ramp = ((conc.unsqueeze(-1) - self.lo) / self.width).clamp(0.0, 1.0)
        return (ramp * self.step).sum(-1).clamp(max=self.cap)

    def forward(self, conc):
        """(..., P) concentrations -> ((...) AQI, (..., P) sub-indices)."""
        sub = self.sub_index(conc)
        hard = sub.max(dim=-1).values
        soft = self.tau * torch.logsumexp(sub / self.tau, dim=-1)
        # value = the exact CPCB maximum; gradient = the smooth maximum's
        return hard.detach() + soft - soft.detach(), sub


# Every pollutant the breakpoint table covers and the daily table carries...
ALL_POLLUTANTS = [p for p in aqi_calculator.BREAKPOINTS if p in daily.columns]
# ...and the subset the models are actually fed. config.POLLUTANTS leaves NH3 out, so the
# head can only build the index from these. The check below measures what that costs.
HEAD_POLLUTANTS = [p for p in aqi_calculator.BREAKPOINTS if p in variates["fair"]]


def cpcb_error(pollutants, sample):
    with torch.no_grad():
        got, _ = CPCBSubIndex(pollutants)(torch.tensor(sample[pollutants].to_numpy(dtype=np.float32)))
    truth = sample["AQI"].to_numpy(dtype=np.float32)
    return float(np.abs(got.numpy() - truth).mean()), float(np.corrcoef(got.numpy(), truth)[0, 1])


_check = daily.dropna(subset=ALL_POLLUTANTS + ["AQI"])
if len(_check) > 4000:
    _check = _check.sample(4000, random_state=0)
_err_all, _corr_all = cpcb_error(ALL_POLLUTANTS, _check)
_err_head, _corr_head = cpcb_error(HEAD_POLLUTANTS, _check)
print(f"differentiable CPCB layer against the dataset's own AQI, over {len(_check):,} days")
print(f"  all {len(ALL_POLLUTANTS)} pollutants {ALL_POLLUTANTS}:")
print(f"      MAE {_err_all:5.2f} AQI points, correlation {_corr_all:.4f}")
print(f"  the {len(HEAD_POLLUTANTS)} the models read {HEAD_POLLUTANTS}:")
print(f"      MAE {_err_head:5.2f} AQI points, correlation {_corr_head:.4f}")
if HEAD_POLLUTANTS != ALL_POLLUTANTS:
    print(f"  (left out: {sorted(set(ALL_POLLUTANTS) - set(HEAD_POLLUTANTS))} -- not in "
          f"config.POLLUTANTS, so no model in this repository sees it)")
if _err_head > 25.0:
    raise RuntimeError(f"the differentiable CPCB layer does not reproduce the dataset's AQI "
                       f"(MAE {_err_head:.1f}); stopping rather than training on a wrong formula")

## B · The models

`SubIndexModel` reuses the all-token encoder exactly as the other models use it, and
changes only the output layer: it predicts a multiplicative correction to **today's
pollutant concentrations**, and the AQI is read off the CPCB formula. The correction head is
zero-initialised, so the model begins as persistence-of-concentrations — which maps
through the formula to persistence-of-AQI — and must earn every departure from it. That
mirrors the anchoring already on the ladder, one level lower down.

In [ ]:
class SubIndexModel(multires.FinalModel):
    """All-token encoder; the CPCB sub-index formula as the output layer."""

    def __init__(self, n_variates, lookback, horizon, target_index, poll_indices, pollutants,
                 d_model=128, n_heads=8, n_layers=2, dropout=0.3, limit=3.0):
        super().__init__(n_variates, lookback, horizon, target_index, anchor_indices=None,
                         mixture=("mean",), n_channels=0, d_model=d_model, n_heads=n_heads,
                         n_layers=n_layers, dropout=dropout)
        if len(poll_indices) != len(pollutants):
            raise ValueError("one input index per pollutant the head builds the index from")
        self.poll_indices = list(poll_indices)
        self.n_poll = len(self.poll_indices)
        self.limit = float(limit)
        self.expert = nn.Linear(n_variates * d_model, horizon * self.n_poll)
        nn.init.zeros_(self.expert.weight)
        nn.init.zeros_(self.expert.bias)
        self.cpcb = CPCBSubIndex(pollutants)

    def forward(self, inputs):
        encoded, _, _, _ = self._encode(inputs)
        window, _ = self.split(inputs)
        delta = self.expert(self.drop(encoded.flatten(1))).view(-1, self.horizon, self.n_poll)
        today = window[:, -1, self.poll_indices].unsqueeze(1).clamp(min=0.0)
        # multiplicative and positive: zero-init -> exp(0) = 1 -> today's concentrations
        conc = today * torch.exp(delta.clamp(-self.limit, self.limit))
        aqi, _ = self.cpcb(conc)
        return aqi


anchor_indices = [variates["fair"].index(c) for c in ANCHOR_COLUMNS]
poll_indices = [variates["fair"].index(p) for p in HEAD_POLLUTANTS]
n_hourly = len(multires.HOURLY_CHANNELS)
size = dict(d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT)
n_fair = len(variates["fair"])


def all_token(anchors=False, hours=False, mixture=multires.MIXTURE):
    return multires.FinalModel(n_fair, LOOKBACK, HORIZON, target["fair"],
                               anchor_indices=anchor_indices if anchors else None,
                               mixture=mixture, n_channels=n_hourly if hours else 0, **size)


# name -> (feature set, reads the 72 raw hours too, how it is built)
BUILDERS = {
    # --- identical inputs, different architectures: this is the fair comparison
    "itransformer_fair": ("fair", False,
                          lambda: itransformer.ITransformer(n_fair, LOOKBACK, HORIZON, target["fair"], **size)),
    "lstm_fair": ("fair", False,
                  lambda: deep_baselines.LSTMForecaster(n_fair, HORIZON, target["fair"], dropout=DROPOUT)),
    "head_fair": ("fair", False, lambda: all_token()),
    # --- the full model on the same feature set, plus the 72 raw hourly tokens
    "dextra_fair": ("fair", True, lambda: all_token(anchors=True, hours=True)),
    # --- the new rung
    "dsi": ("fair", False,
            lambda: SubIndexModel(n_fair, LOOKBACK, HORIZON, target["fair"],
                                  poll_indices, HEAD_POLLUTANTS, **size)),
}
FAIR_FAMILY = ["itransformer_fair", "lstm_fair", "head_fair", "dextra_fair"]
SUBINDEX_FAMILY = ["dsi"]
ANCHORED_MODELS = {"dextra_fair"}

rows = [{"model": name, "reads": kind + (" + 72 hours" if hours else ""),
         "parameters": itransformer.count_parameters(build())}
        for name, (kind, hours, build) in BUILDERS.items()]
print(pd.DataFrame(rows).set_index("model").to_string())
print(f"\nevery model above reads the same {n_fair} variates "
      f"({len(SHAPE_COLUMNS)} of them the hourly shape the gradient boosting also gets)")

## C · The harness

Identical to notebook 08's: the same folds, the same seeds, the same early stopping and
the same per-city ledger, so the numbers can be compared directly with notebook 08's.

In [ ]:
# What this session computed, so the analysis below does not download back what is
# already in memory -- and so it still works in a run that is not saving anything.
COMPUTED = {}


def save_run(name, result, frames=None):
    for filename, frame in (frames or {}).items():
        if filename == f"{name}.parquet":
            COMPUTED[name] = frame
    if not SAVE_TO_HF:
        return
    with tempfile.TemporaryDirectory() as tmp:
        attachments = {}
        for filename, frame in (frames or {}).items():
            attachments[filename] = Path(tmp) / filename
            frame.to_parquet(attachments[filename], index=False)
        registry.record(name, result, attachments=attachments)


def stop_here(message="time budget reached"):
    print(f"[paused] {message} -- Runtime -> Disconnect and delete runtime, then Run all")
    return True


def best_epoch(history):
    return min(history, key=lambda h: h["val_rmse"])["epoch"] if history else 0


def fold_windows(city, names):
    labelled = daily.assign(split=assemble.leave_one_out_split(daily, city))
    windows = {}
    for kind in dict.fromkeys(BUILDERS[name][0] for name in names):
        X, y, index = sequences.build_sequences(labelled, features=FEATURES[kind])
        windows[kind] = {"X": X, "y": y, "index": index}
        if any(BUILDERS[name][1] for name in names if BUILDERS[name][0] == kind):
            windows[kind]["packed"] = multires.pack(X, multires.attach_hourly(index, hourly_table))
    return windows


def run_family(names, windows):
    """Train each model with every seed. Returns (frame, runs); frame is None if time ran out."""
    runs, frames = [], []
    for kind, cut in windows.items():
        X, y, index = cut["X"], cut["y"], cut["index"]
        train_rows, val_rows, test_rows = ((index.split == s).to_numpy() for s in ("train", "val", "test_loco"))
        keep = val_rows | test_rows
        part = {}
        for name in [n for n in names if BUILDERS[n][0] == kind]:
            inputs = cut["packed"] if BUILDERS[name][1] else X
            val_loader = itransformer.make_loader(inputs[val_rows], y[val_rows], BATCH_SIZE * 2)
            for seed in SEEDS:
                itransformer.set_seed(seed)
                model, history, outcome = training.train(
                    BUILDERS[name][2](),
                    itransformer.make_loader(inputs[train_rows], y[train_rows], BATCH_SIZE, shuffle=True),
                    val_loader, device, epochs=EPOCHS, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
                    patience=PATIENCE, deadline=deadline, save_finished=False, log=lambda *a: None)
                if outcome == "out_of_time":
                    return None, None
                part[f"{name}_seed{seed}"] = itransformer.predict(model, inputs[keep], device)
                runs.append({"model": name, "seed": seed, "outcome": outcome,
                             "best_epoch": best_epoch(history), "epochs_run": len(history),
                             "val_rmse": min(h["val_rmse"] for h in history)})
        frames.append(gap.long_frame(index[keep], y[keep], X[keep, -1, target[kind]], part))
    frame = frames[0]
    for other in frames[1:]:
        frame = frame.merge(other.drop(columns=["split", "y", "persistence"]), on=KEYS, how="inner")
    return frame, runs

## D · The fair comparison, fold by fold

Fifteen folds. Each is recorded the moment it finishes, so a disconnect costs only the
fold in progress.

In [ ]:
stopped = False
for city in cities:
    run = fair_run(city)
    if registry.done(run):
        print(f"[done] {run} already finished")
        continue
    if stopped or time.monotonic() > deadline:
        stopped = stop_here()
        break
    print(f"\n> {run}")
    started = time.monotonic()
    frame, runs = run_family(FAIR_FAMILY, fold_windows(city, FAIR_FAMILY))
    if frame is None:
        stopped = stop_here("time budget reached mid-fold; this fold starts again next session")
        break
    minutes = round((time.monotonic() - started) / 60, 2)
    save_run(run, {"city": city, "runs": runs, "windows": len(frame) // HORIZON, "minutes": minutes},
             frames={f"{run}.parquet": frame})
    summary = pd.DataFrame(runs).groupby("model").val_rmse.mean().round(2).to_dict()
    print(f"  validation RMSE {summary};  recorded, {minutes} min")

## E · The CPCB sub-index head, fold by fold

In [ ]:
for city in cities:
    run = subindex_run(city)
    if registry.done(run):
        print(f"[done] {run} already finished")
        continue
    if stopped or time.monotonic() > deadline:
        stopped = stop_here()
        break
    print(f"\n> {run}")
    started = time.monotonic()
    frame, runs = run_family(SUBINDEX_FAMILY, fold_windows(city, SUBINDEX_FAMILY))
    if frame is None:
        stopped = stop_here("time budget reached mid-fold; this fold starts again next session")
        break
    minutes = round((time.monotonic() - started) / 60, 2)
    save_run(run, {"city": city, "runs": runs, "windows": len(frame) // HORIZON, "minutes": minutes},
             frames={f"{run}.parquet": frame})
    summary = pd.DataFrame(runs).groupby("model").val_rmse.mean().round(2).to_dict()
    print(f"  validation RMSE {summary};  recorded, {minutes} min")

## F · What it shows

Only the folds that finished are scored, so this is readable mid-way through.

In [ ]:
def load_frame(reg, name):
    """This session's own frame if we have it, otherwise the recorded one."""
    if name in COMPUTED:
        return COMPUTED[name]
    return by_day(pd.read_parquet(io.BytesIO(store.read_bytes(reg.attachment(f"{name}.parquet")))))


def have(name):
    return name in COMPUTED or registry.done(name)


done_cities = [c for c in cities if have(fair_run(c))]
scores = verdict = pd.DataFrame()
if not done_cities:
    print("No fold has finished yet. Run the cells above; this one is safe to re-run after.")
else:
    parts = []
    for city in done_cities:
        piece = load_frame(registry, fair_run(city))
        if have(subindex_run(city)):
            extra = load_frame(registry, subindex_run(city))
            piece = piece.merge(extra.drop(columns=["split", "y", "persistence"]), on=KEYS, how="inner")
        if registry08.done("loco-" + city.replace(" ", "_") + "-models"):
            old = load_frame(registry08, "loco-" + city.replace(" ", "_") + "-models")
            keep = [c for c in old.columns if c.startswith(("dextra_seed", "control_seed"))]
            piece = piece.merge(old[KEYS + keep], on=KEYS, how="left")
        parts.append(piece.assign(fold=city))
    everything = pd.concat(parts, ignore_index=True)
    print(f"{len(done_cities)} folds scored: {', '.join(done_cities)}")

    LINEUP = [("Persistence", ["persistence"])]
    for name in ["itransformer_fair", "lstm_fair", "head_fair", "dextra_fair", "dsi"]:
        columns = [c for c in seeded(name) if c in everything.columns]
        if columns:
            LINEUP.append((name, columns))
    for label, name in (("dextra (notebook 08)", "dextra"), ("control (notebook 08)", "control")):
        columns = [c for c in seeded(name) if c in everything.columns]
        if columns:
            LINEUP.append((label, columns))

    # Every forecaster is scored on exactly the same rows. Notebook 08's columns are
    # merged in, and a row it does not carry would otherwise be NaN for those models
    # alone -- which both breaks average precision and makes the pairing uneven.
    compared = [c for _, columns in LINEUP for c in columns]
    tested = everything[everything.split == "test_loco"]
    before = len(tested)
    tested = tested.dropna(subset=compared)
    if len(tested) < before:
        print(f"  {before - len(tested):,} of {before:,} held-out rows dropped so that every "
              f"forecaster is scored on the same rows")

    rows = []
    for label, columns in LINEUP:
        rows.append({"forecaster": label,
                     "AP": gap.average_precision(tested, columns),
                     "RMSE": gap.rmse(tested, columns),
                     "next-day RMSE": gap.rmse(tested[tested.horizon == 1], columns)})
    scores = pd.DataFrame(rows).set_index("forecaster")
    print("\nHeld-out cities, every model on the same inputs:")
    print(scores.round(3).to_string())

    # the comparisons the paper actually needs
    pairs = [("head_fair", "itransformer_fair", "our architecture vs iTransformer, identical inputs"),
             ("head_fair", "lstm_fair", "our architecture vs LSTM, identical inputs"),
             ("dextra_fair", "head_fair", "anchors + 72 raw hours, on top"),
             ("dsi", "head_fair", "the CPCB sub-index head, on top")]
    rows = []
    for a, b, question in pairs:
        ca, cb = [c for c in seeded(a) if c in tested.columns], [c for c in seeded(b) if c in tested.columns]
        if not ca or not cb:
            continue
        d_ap, lo_ap, hi_ap, p_ap = gap.paired_ap_difference(tested, ca, cb)
        d_rm, lo_rm, hi_rm, p_rm = gap.paired_rmse_difference(tested, ca, cb)
        rows.append({"question": question, "AP difference": d_ap,
                     "AP 95%": f"[{lo_ap:+.3f}, {hi_ap:+.3f}]", "AP p": p_ap,
                     "RMSE difference": d_rm, "RMSE 95%": f"[{lo_rm:+.3f}, {hi_rm:+.3f}]", "RMSE p": p_rm})
    if rows:
        verdict = pd.DataFrame(rows).set_index("question")
        print("\nThe comparisons the paper needs (first minus second):")
        print(verdict.round(4).to_string())
        print("\nAn interval spanning zero is a tie. Intervals are per-comparison and "
              "uncorrected; with four comparisons, read them together.")

In [ ]:
# ── save ────────────────────────────────────────────────────────────────────
if SAVE_TO_HF and len(scores):
    with tempfile.TemporaryDirectory() as tmp:
        for name, frame in (("fair_scores", scores), ("fair_verdict", verdict)):
            if not len(frame):
                continue
            path = Path(tmp) / f"{name}.csv"
            frame.to_csv(path)
            store.upload_file(str(path), f"processed/{name}.csv",
                              f"Notebook 11: {name.replace('_', ' ')}")
            print(f"  uploaded processed/{name}.csv")
    print("\nDone. Tell Claude the run finished and it will redo the paper's figures and tables.")
elif len(scores):
    print("\nNot saving (SAVE_TO_HF is off).")